In [2]:
import numpy as np
import pandas as pd
import math
import scipy
import sympy as sp
import random
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt
from math import pi
from scipy.interpolate import NearestNDInterpolator

%matplotlib inline

#floris simulation modules
from floris.tools.wind_rose import WindRose
from floris.tools import FlorisInterface
import floris.tools.visualization as wakeviz
from floris.turbine_library import TurbineInterface, TurbineLibrary
from floris.tools.visualization import (
    calculate_horizontal_plane_with_turbines,
    visualize_cut_plane,
)



In [39]:
rhow = 1.226
D = 52
A = round(((math.pi * D**2)/4))
V = 12
V_3 = V**3

In [40]:
P_in = (round(((0.5 * rhow * A * V_3)))/1000)
P_in

2249.877

In [41]:
p_out = [0, 0, 0, 0,
         5, 15, 30, 49, 67, 97,
         127, 162, 197, 244, 290, 350,
         410, 475, 539, 600, 660, 710,
         751, 784, 816, 840, 850, 850,
         850, 850, 850, 850, 850, 850,
         850, 850, 850, 850, 850, 850,
         850, 850, 850, 850, 850, 850,
         850, 850, 850,
         0.0, 0.0]
len(p_out)

51

In [42]:
wind_v_list = []
for i in range(2, 53):
    i = i/2
    wind_v_list.append(i)

print(wind_v_list)
print(len(wind_v_list))

[1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5, 14.0, 14.5, 15.0, 15.5, 16.0, 16.5, 17.0, 17.5, 18.0, 18.5, 19.0, 19.5, 20.0, 20.5, 21.0, 21.5, 22.0, 22.5, 23.0, 23.5, 24.0, 24.5, 25.0, 25.5, 26.0]
51


In [43]:
def calculate_p_in(rhow, A, v):
    """
    Calculates the power in (p_in) based on the given formula.

    Parameters:
    - rhow: Density
    - A: Area
    - v: List of velocities

    Returns:
    - List of power in (p_in)
    """
    p_in_values = []
    for velocity in v:
        V_3 = velocity ** 3
        p_in = round(0.5 * rhow * A * V_3) / 1000
        p_in_values.append(p_in)
    return p_in_values
    
p_in = calculate_p_in(rhow, A, wind_v_list)
print("p_in values:", p_in)

p_in values: [1.302, 4.394, 10.416, 20.344, 35.154, 55.824, 83.329, 118.646, 162.752, 216.622, 281.235, 357.565, 446.59, 549.286, 666.63, 799.598, 949.167, 1116.313, 1302.012, 1507.242, 1732.978, 1980.198, 2249.877, 2542.992, 2860.52, 3203.438, 3572.721, 3969.346, 4394.29, 4848.53, 5333.041, 5848.801, 6396.785, 6977.971, 7593.334, 8243.852, 8930.5, 9654.256, 10416.096, 11216.996, 12057.933, 12939.884, 13863.824, 14830.73, 15841.58, 16897.349, 17999.014, 19147.551, 20343.938, 21589.149, 22884.163]


In [44]:
def Cp_values(v, p_in, p_out):
        cp_values = []
        for i in range(len(v)):
            if p_out[i] != 0:
                cp = round((p_out[i] / p_in[i]), 3)
                cp_values.append(cp)
            else:
                cp_values.append(float(0))  # handle division by zero
        return cp_values
            
cp_values = Cp_values(wind_v_list, p_in, p_out)
CP_Values = cp_values
print("Cp values:", cp_values)
print("Length of Cp values:", len(cp_values))
print("Length of wind_v_list:", len(wind_v_list))

Cp values: [0.0, 0.0, 0.0, 0.0, 0.142, 0.269, 0.36, 0.413, 0.412, 0.448, 0.452, 0.453, 0.441, 0.444, 0.435, 0.438, 0.432, 0.426, 0.414, 0.398, 0.381, 0.359, 0.334, 0.308, 0.285, 0.262, 0.238, 0.214, 0.193, 0.175, 0.159, 0.145, 0.133, 0.122, 0.112, 0.103, 0.095, 0.088, 0.082, 0.076, 0.07, 0.066, 0.061, 0.057, 0.054, 0.05, 0.047, 0.044, 0.042, 0.0, 0.0]
Length of Cp values: 51
Length of wind_v_list: 51


In [45]:
def newton_raphson_method(func, deriv_func, initial_guess):
    a = initial_guess
    while True:
        deriv = deriv_func(a)
        if deriv == 0:
            break
        a_new = a - func(a) / deriv
        if abs(a_new - a) < 1e-6:  # a small number for the precision of the approximation
            break
        a = a_new
    return a

a = sp.symbols('a')


a_value = []
for i in CP_Values:
    func = 4*a*(1 - a)**2 - i
    deriv_func = sp.diff(func, a)

    func = sp.lambdify(a, func)
    deriv_func = sp.lambdify(a, deriv_func)

    initial_guess = 0.5
    a_result = newton_raphson_method(func, deriv_func, initial_guess)  # Use a different variable here
    a_value.append(a_result)

print(f"The value of Ct is {CP_Values}")

def CT(a):
    return 4*a*(1 - a)

CT_value = [CT(value) for value in a_value]

CT_value_rounded = [round(value, 3) for value in CT_value]
print(f"The value of CT is {CT_value_rounded}")

print(len(CT_value_rounded))
print(len(CP_Values))

The value of Ct is [0.0, 0.0, 0.0, 0.0, 0.142, 0.269, 0.36, 0.413, 0.412, 0.448, 0.452, 0.453, 0.441, 0.444, 0.435, 0.438, 0.432, 0.426, 0.414, 0.398, 0.381, 0.359, 0.334, 0.308, 0.285, 0.262, 0.238, 0.214, 0.193, 0.175, 0.159, 0.145, 0.133, 0.122, 0.112, 0.103, 0.095, 0.088, 0.082, 0.076, 0.07, 0.066, 0.061, 0.057, 0.054, 0.05, 0.047, 0.044, 0.042, 0.0, 0.0]
The value of CT is [0.0, 0.0, 0.0, 0.0, 0.669, 0.86, 0.944, 0.976, 0.976, 0.991, 0.992, 0.992, 0.988, 0.989, 0.986, 0.987, 0.985, 0.982, 0.977, 0.968, 0.958, 0.943, 0.923, 0.9, 0.877, 0.852, 0.823, 0.79, 0.759, 0.729, 0.701, 0.675, 0.651, 0.627, 0.604, 0.582, 0.562, 0.543, 0.526, 0.508, 0.49, 0.477, 0.46, 0.446, 0.435, 0.42, 0.408, 0.395, 0.387, 0.0, 0.0]
51
51
